# numpy + scipy — task-indexed cheatsheet


Look up the idiom you need by **task**. Each entry: question → canonical pattern → why + common mistake.

Covers: array creation, indexing, broadcasting, vectorised math, reductions, linear algebra, random number generation, and the bits of `scipy.stats` you keep needing for ML work (moments, hypothesis tests, distributions).


---
## Setup

Run this once.


### Setup — run me first


In [1]:
import numpy as np
from scipy import stats

# Reproducible RNG for every example.
rng = np.random.default_rng(0)

---
## 1. Creating arrays

Pre-allocate then fill. Avoid building a list and converting at the end — you usually know the shape upfront.


### How do I create an array of zeros / ones / a constant?

`np.zeros(shape)`, `np.ones(shape)`, `np.full(shape, value)`. Shape can be an int (1D) or a tuple (multi-dim).


In [2]:
print(np.zeros(5))                       # 1D, all zeros
print(np.ones((2, 3)))                   # 2D, all ones
print(np.full((2, 2), -1.0))             # 2D, all -1.0

[0. 0. 0. 0. 0.]
[[1. 1. 1.]
 [1. 1. 1.]]
[[-1. -1.]
 [-1. -1.]]


*Why preallocate*: building a Python list and converting with `np.array(list)` is much slower for large arrays. *Common mistake*: `np.zeros(2, 3)` (no tuple) — interprets as `dtype=3`.


### How do I create an evenly-spaced range?

`np.arange(start, stop, step)` for integer-like steps; `np.linspace(start, stop, n)` for n equally-spaced floats including both endpoints.


In [3]:
print(np.arange(0, 10, 2))               # 0, 2, 4, 6, 8 (stop exclusive)
print(np.linspace(0, 1, 5))              # 0.0, 0.25, 0.5, 0.75, 1.0 (both inclusive)

[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]


*When to use which*: arange for indices and integer steps; linspace for plotting grids and any "evenly partition this interval" need. *Common mistake*: relying on `arange(0, 1, 0.1)` — float accumulation gives 11 entries on some platforms; use `linspace(0, 1, 11)` instead.


### How do I create an array with the same shape as another?

`np.zeros_like(x)` / `np.ones_like(x)` / `np.full_like(x, value)`. Inherits dtype too.


In [4]:
x = np.array([[1, 2], [3, 4]])
print(np.zeros_like(x))
print(np.full_like(x, fill_value=99.0, dtype=float))   # override dtype

[[0 0]
 [0 0]]
[[99. 99.]
 [99. 99.]]


*Why _like*: less error-prone than typing the shape manually, and dtype matches automatically.


---
## 2. Indexing and slicing

Arrays support three indexing modes: slicing (returns a *view*), fancy indexing (returns a *copy*), boolean masks (returns a *copy*). The view-vs-copy distinction matters for mutation.


### How do I select a row / column / submatrix?

Use commas for separate dimensions. `arr[i]` is row i; `arr[:, j]` is column j; `arr[i:j, k:l]` is a block.


In [5]:
A = np.arange(12).reshape(3, 4)
print(A)
print(A[0])              # first row
print(A[:, 2])           # third column
print(A[1:3, 0:2])       # block: rows 1-2, columns 0-1

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[0 1 2 3]
[ 2  6 10]
[[4 5]
 [8 9]]


*Common mistake*: `A[i][j]` and `A[i, j]` look equivalent for reads but `A[i][j] = ...` may not modify `A` because `A[i]` returned a view that gets overwritten. Always use `A[i, j]` for assignment.


### How do I select rows where a condition is True?

Boolean mask. `arr[mask]` keeps the rows / elements where `mask` is True.


In [6]:
x = np.array([10, 20, 30, 40, 50])
print(x[x > 20])                 # [30, 40, 50]
print(x[(x > 15) & (x < 45)])    # use & not `and` for arrays

[30 40 50]
[20 30 40]


*Critical detail*: arrays use `&`, `|`, `~` for elementwise boolean ops, not `and`, `or`, `not`. Python's `and` doesn't work on arrays — it tries to evaluate the array's truth value and raises `ValueError`.


### How do I pick specific indices (not a slice)?

Pass a list or array of integer indices. This is *fancy indexing* — returns a copy.


In [7]:
x = np.array([10, 20, 30, 40, 50])
print(x[[0, 2, 4]])              # pick positions 0, 2, 4
print(x[np.argsort(-x)[:3]])     # pick top-3 by value (descending)

[10 30 50]
[50 40 30]


*Why this is useful*: combined with `argsort`/`argmax`, lets you pick top-k items in one line. *Common mistake*: assuming the result is a view — fancy indexing always copies, so modifying the result doesn't change the original.


---
## 3. Reductions (sum, mean, max, etc.)

All numpy reductions take an `axis=` argument. The single trickiest thing: `axis=0` reduces *rows* (output is shape (cols,)); `axis=1` reduces *columns* (output is shape (rows,)).


### How do I sum / mean / std along a specific axis?

`arr.sum(axis=...)`. `axis=None` (default) collapses everything to a scalar.


In [8]:
A = np.arange(12).reshape(3, 4)
print('sum all:        ', A.sum())             # scalar
print('sum per column: ', A.sum(axis=0))       # length 4 (one per column)
print('sum per row:    ', A.sum(axis=1))       # length 3 (one per row)

sum all:         66
sum per column:  [12 15 18 21]
sum per row:     [ 6 22 38]


*Memorise this rule*: `axis=k` removes axis k. Shape (3, 4) with `axis=0` → shape (4,); `axis=1` → shape (3,). *Common mistake*: confusing axis 0 (rows) and axis 1 (cols) — reread the rule above when in doubt.


### How do I find the index of the max / min?

`np.argmax` / `np.argmin`. Same axis convention as the reductions.


In [9]:
x = np.array([3, 1, 4, 1, 5, 9, 2])
print(x.argmax(), x.max())       # 5, 9
print(np.argsort(x)[::-1][:3])    # top-3 indices, descending

5 9
[5 4 2]


*Why argsort*: gives you the *full* ranking, not just the max. `[::-1]` reverses to descending; `[:k]` takes the top k.


### How do I get the percentiles / quantiles?

`np.percentile(arr, q)` (q in 0-100) or `np.quantile(arr, q)` (q in 0-1).


In [10]:
x = rng.normal(0, 1, 1000)
print('p10, p50, p90:', np.percentile(x, [10, 50, 90]).round(3))
print('q05, q95:    ', np.quantile(x, [0.05, 0.95]).round(3))

p10, p50, p90: [-1.256 -0.075  1.23 ]
q05, q95:     [-1.556  1.514]


*Common mistake*: `percentile(x, 0.5)` returns the 0.5th percentile (close to the minimum), not the median. Use `quantile` for the 0-1 convention.


---
## 4. Vectorisation — replacing Python loops

If you find yourself writing `for i in range(len(arr))`, stop and look for the vectorised version. Vectorised operations are 10-100× faster and usually clearer.


### How do I apply elementwise math?

Just use the operator. `a + b`, `np.sqrt(a)`, `np.log(a)`, etc. all work elementwise on arrays.


In [11]:
x = np.array([1, 2, 3, 4])
print(x ** 2)            # square each
print(np.sqrt(x))        # sqrt each
print(np.log(x + 1))     # vectorised math chain

[ 1  4  9 16]
[1.         1.41421356 1.73205081 2.        ]
[0.69314718 1.09861229 1.38629436 1.60943791]


*Why it works*: numpy ops are implemented in C and run at near-machine speed. *Common mistake*: writing `[xi**2 for xi in x]` — same result, 50× slower.


### How do I apply a conditional (if/else) elementwise?

`np.where(condition, value_if_true, value_if_false)`. All three can be arrays or scalars.


In [12]:
x = np.array([1, -2, 3, -4, 5])
print(np.where(x > 0, x, 0))     # ReLU: max(0, x)
print(np.where(x > 0, 'pos', 'neg'))  # branching to strings

[1 0 3 0 5]
['pos' 'neg' 'pos' 'neg' 'pos']


*When to use*: any "if this then that, else other" elementwise. *Common mistake*: passing a 0-D scalar where you meant an array — works but doesn't broadcast as expected.


### How do I apply a multi-way conditional (multiple branches)?

`np.select([cond1, cond2, ...], [val1, val2, ...], default=...)`.


In [13]:
x = np.array([5, 25, 75, 95])
buckets = np.select(
    [x < 25, x < 50, x < 75],
    ['low', 'mid_low', 'mid_high'],
    default='high',
)
print(buckets)

['low' 'mid_low' 'high' 'high']


*When to use*: cleaner than nested `np.where`. Conditions are checked in order, first match wins.


### How do I clip values to a range?

`np.clip(arr, lo, hi)` — values outside [lo, hi] are replaced with the nearest endpoint.


In [14]:
x = np.array([-2, -1, 0, 1, 2])
print(np.clip(x, -1, 1))        # values stay in [-1, 1]
print(np.clip(rng.normal(0, 1, 5), -2, 2))   # winsorise outliers

[-1 -1  0  1  1]
[ 1.18390191  0.30382685  0.19205435  0.26594345 -1.36615788]


*Common use*: bounding probabilities to (eps, 1-eps) before log-loss; winsorising outliers before fitting a linear model.


---
## 5. Broadcasting — operations on different-shaped arrays

Broadcasting lets numpy combine arrays of different shapes without copying. **Rule**: align trailing dimensions; sizes must match or be 1. Otherwise it errors.


### How does adding a scalar to an array work?

The scalar is broadcast across every element. Shape (n,) + scalar → shape (n,).


In [15]:
print(np.array([1, 2, 3]) + 10)      # [11, 12, 13]

[11 12 13]


*Why this matters*: the same rule generalises to higher dimensions. Once you understand `(n,) + scalar`, the rest follows.


### How do I add a 1D vector to every row of a 2D matrix?

Just `+`. The (n,) vector broadcasts against the (rows, n) matrix.


In [16]:
A = np.arange(12).reshape(3, 4)
v = np.array([10, 20, 30, 40])
print(A + v)                     # adds v to each row

[[10 21 32 43]
 [14 25 36 47]
 [18 29 40 51]]


*Why it works*: the vector's shape (4,) is treated as (1, 4), then broadcast to (3, 4). *Common mistake*: trying to add a (3,) vector — fails because the trailing dimension (3) doesn't match the matrix's trailing dimension (4).


### How do I add a 1D vector to every COLUMN (not row) of a 2D matrix?

Reshape the vector to a column with `[:, None]` (or `.reshape(-1, 1)`).


In [17]:
A = np.arange(12).reshape(3, 4)
col_v = np.array([10, 20, 30])[:, None]   # shape (3, 1)
print(A + col_v)                          # adds to each column

[[10 11 12 13]
 [24 25 26 27]
 [38 39 40 41]]


*The trick*: `[:, None]` adds a trailing axis, turning (3,) into (3, 1). Now (3, 1) + (3, 4) broadcasts column-wise. *Common mistake*: forgetting `[:, None]` and getting a shape mismatch.


---
## 6. Linear algebra (just enough)

You probably don't need linear algebra most days, but when you do, two functions cover 90% of cases.


### How do I do matrix multiplication?

`A @ B` (Python operator) or `np.matmul(A, B)`. NOT `A * B` — that's elementwise.


In [18]:
A = np.arange(6).reshape(2, 3)
B = np.arange(12).reshape(3, 4)
print(A @ B)                    # shape (2, 4)

[[20 23 26 29]
 [56 68 80 92]]


*Why @ over .dot()*: identical result, but @ is a real Python operator and reads cleaner. *Common mistake*: `A * B` does Hadamard (elementwise) product when shapes match — silently different from matrix multiplication.


### How do I solve a linear system Ax = b?

`np.linalg.solve(A, b)` for square A. `np.linalg.lstsq(A, b)` for the least-squares solution to over- or under-determined systems.


In [19]:
A = np.array([[2, 1], [1, 3]])
b = np.array([3, 4])
x = np.linalg.solve(A, b)
print('x =', x)
print('A @ x =', A @ x)         # sanity check: should equal b

x = [1. 1.]
A @ x = [3. 4.]


*Why solve over inv*: numerically more stable than computing `np.linalg.inv(A) @ b`. Always prefer `solve` when you can.


### How do I fit a linear regression by hand?

Stack ones for the intercept; use `np.linalg.lstsq`.


In [20]:
n = 50
x = rng.uniform(-1, 1, n)
y = 2 * x + 1 + rng.normal(0, 0.2, n)

X = np.column_stack([np.ones(n), x])    # design matrix with intercept
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
print(f'intercept = {beta[0]:.3f}, slope = {beta[1]:.3f}')

intercept = 1.003, slope = 2.006


*When to use*: when sklearn's overhead (instantiating `LinearRegression`, `.fit()`, `.predict()`) is too heavy for a one-line fit. *Common mistake*: forgetting the intercept column.


---
## 7. Random number generation

**Use `default_rng`**, not the legacy `np.random.seed` API. It's per-instance, reproducible, and won't be affected by other code that touches the global RNG.


### How do I create a reproducible RNG?

`rng = np.random.default_rng(seed)`. Use `rng` for all draws thereafter.


In [21]:
rng_a = np.random.default_rng(42)
print(rng_a.uniform(0, 1, 3))    # always the same numbers
rng_b = np.random.default_rng(42)
print(rng_b.uniform(0, 1, 3))    # identical to rng_a

[0.77395605 0.43887844 0.85859792]
[0.77395605 0.43887844 0.85859792]


*Why default_rng over np.random.seed*: legacy seed is global state — a third-party library calling `np.random.seed(0)` resets your RNG mid-program. Per-instance RNGs are immune.


### How do I draw from common distributions?

Methods on the rng instance: `.uniform`, `.normal`, `.integers`, `.choice`, `.exponential`, `.beta`, etc.


In [22]:
print(rng.uniform(0, 1, 5))                   # uniform on [0, 1)
print(rng.normal(loc=0, scale=1, size=5))     # standard normal
print(rng.integers(0, 100, size=5))           # ints in [0, 100)
print(rng.choice(['a', 'b', 'c'], 5, p=[0.6, 0.3, 0.1]))   # weighted

[0.11908584 0.81793344 0.34552249 0.69176262 0.98846591]
[ 1.22704093  0.37721495  0.20781545 -1.22360358  0.29206254]
[99 87 32 96 55]
['a' 'b' 'b' 'c' 'b']


*Common mistake*: `rng.integers(0, 100)` is **half-open** (max 99). Old `np.random.randint` was the same; the docs are explicit but easy to miss.


### How do I shuffle / permute / resample?

`rng.shuffle(arr)` (in-place), `rng.permutation(arr)` (returns shuffled copy), `rng.choice(arr, n, replace=True)` (bootstrap).


In [23]:
arr = np.arange(5)
print(rng.permutation(arr))               # shuffled copy
print(rng.choice(arr, 5, replace=True))   # bootstrap sample

[3 0 1 4 2]
[2 1 0 3 3]


*Common mistake*: using `rng.shuffle(arr)` and then trying to use the return value — `shuffle` returns `None`. Use `permutation` if you want the result.


---
## 8. scipy.stats — moments and hypothesis tests

Higher moments and the hypothesis tests you reach for when characterising a distribution or comparing two samples.


### How do I compute skew and kurtosis?

`stats.skew(x)` and `stats.kurtosis(x, fisher=True)`. Fisher convention: 0 = normal.


In [24]:
x = rng.normal(0, 1, 10000)
print(f'skew:           {stats.skew(x):+.3f}    (~0 for normal)')
print(f'excess kurtosis: {stats.kurtosis(x, fisher=True):+.3f}    (~0 for normal)')

# Heavier-tailed distribution.
x_t = rng.standard_t(df=4, size=10000)
print(f'\nt-dist skew:           {stats.skew(x_t):+.3f}')
print(f't-dist excess kurtosis: {stats.kurtosis(x_t, fisher=True):+.3f}    (heavy tails > 0)')

skew:           -0.018    (~0 for normal)
excess kurtosis: -0.041    (~0 for normal)

t-dist skew:           +0.149
t-dist excess kurtosis: +8.883    (heavy tails > 0)


*Why fisher=True*: subtracts 3 so 0 is the normal-distribution baseline. Easier to read at a glance. *Common mistake*: comparing kurtosis values across `fisher=True` and `fisher=False` — they differ by 3.


### How do I test if a sample is normally distributed?

Shapiro-Wilk for small n, D'Agostino-Pearson (`stats.normaltest`) for general use.


In [25]:
x = rng.normal(0, 1, 500)
stat, p = stats.normaltest(x)
print(f'normaltest p = {p:.3f}    (>0.05 ⇒ can\'t reject normality)')

normaltest p = 0.027    (>0.05 ⇒ can't reject normality)


*When to use*: deciding whether mean-squared losses are appropriate, whether t-tests are valid. *Common mistake*: with very large samples, normality tests reject for negligible deviations from normal — eyeball the histogram too.


### How do I test if two samples have the same distribution?

Kolmogorov-Smirnov: `stats.ks_2samp(x, y)`. Returns a statistic and p-value.


In [26]:
x = rng.normal(0, 1, 500)
y = rng.normal(0.2, 1, 500)        # slightly shifted
stat, p = stats.ks_2samp(x, y)
print(f'KS stat = {stat:.4f}, p = {p:.4f}')

KS stat = 0.1080, p = 0.0058


*When to use*: distribution shift detection (train vs test distribution); comparing two model's predicted-score distributions. *Why KS over t-test*: KS doesn't assume normality and detects any kind of distribution difference, not just mean shifts.


### How do I test if two samples have the same mean?

Two-sample t-test: `stats.ttest_ind(x, y)`. Use `equal_var=False` (Welch's t-test) when the variances might differ.


In [27]:
x = rng.normal(0, 1, 100)
y = rng.normal(0.5, 1, 100)
t, p = stats.ttest_ind(x, y, equal_var=False)
print(f't = {t:.3f}, p = {p:.4f}')

t = -4.419, p = 0.0000


*Why Welch's*: equal-variance t-test makes a strong assumption that's rarely justified. Welch's costs nothing extra and is robust.


---
## 9. scipy.stats — working with distributions

Each distribution is an object with `.pdf`, `.cdf`, `.ppf` (inverse CDF), `.rvs` (sample). Same API for normal, beta, gamma, t, etc.


### How do I evaluate a normal CDF?

`stats.norm.cdf(x, loc=mean, scale=std)`. Returns P(X ≤ x).


In [28]:
print(stats.norm.cdf(0))                       # 0.5 — symmetric
print(stats.norm.cdf(1.96))                    # ~0.975 — the 95% quantile
print(stats.norm(loc=10, scale=2).cdf(12))     # P(X ≤ 12) for N(10, 4)

0.5
0.9750021048517795
0.8413447460685429


*Why loc/scale*: the standard normal is `loc=0, scale=1`; any other normal is parameterised by mean and std. *Common mistake*: passing variance instead of standard deviation — `scale` is the **std**.


### How do I find the value at a given quantile (inverse CDF)?

`stats.norm.ppf(q)`. Inverse of `cdf`: `cdf(ppf(q)) = q`.


In [29]:
print(stats.norm.ppf(0.025))    # -1.96, lower 2.5%
print(stats.norm.ppf(0.975))    # +1.96, upper 2.5%
print(stats.norm.ppf(0.5))      # 0, the median

-1.9599639845400545
1.959963984540054
0.0


*Common use*: building a 95% confidence interval requires `ppf(0.025)` and `ppf(0.975)`.


### How do I sample from a non-uniform distribution?

`distribution.rvs(size, random_state=rng)`. Or use the rng's method directly.


In [30]:
samples = stats.beta(a=2, b=5).rvs(size=10, random_state=rng)
print(samples.round(3))

[0.451 0.449 0.25  0.206 0.082 0.224 0.214 0.103 0.411 0.265]


*When to use which*: rng methods are fast for common distributions. scipy distribution objects are needed when you also want `cdf`, `ppf` etc.
